In [ ]:
pip install open_clip_torch

In [ ]:
import torch
from PIL import Image
import open_clip
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Initialize the model, preprocessing function and tokenizer

In [ ]:
from clip_zeroshot import build_and_cache_text_features, build_and_cache_image_features, top_k_accuracy, load_cached_features

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
model.eval()  # model in train mode by default
model.to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-16')

# Probe 1 - Binnig

## Binning Probe on CLIP Zero Shot



### Prepare the dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("axiong/imagenet-r")

In [ ]:
print(ds)
print(ds['test'].features)
print(ds["test"][0])
print(len(set(ds["test"]["class_name"])))

In [ ]:
unique_pairs = sorted(set(zip(ds['test']["wnid"], ds['test']["class_name"])))
r_wnids = [pair[0] for pair in unique_pairs]
r_class_names = [pair[1].replace('_', ' ') for pair in unique_pairs]

print(len(r_wnids))
print(r_class_names[:5])

In [ ]:
wnid_to_r_index = {wnid: i for i, wnid in enumerate(r_wnids)}

### Prepare the Text Features

In [ ]:
from imagenet_classes import IMAGENET_TEMPLATES

In [ ]:
r_text_features = build_and_cache_text_features(model, tokenizer, r_class_names, IMAGENET_TEMPLATES, device, "./features", "imagenet_r_text_features")

### Build the dataloader

In [ ]:
class ImageNetRDataset(Dataset):
    def __init__(self, dataset, preprocess, wnid_to_index):
        self.dataset = dataset
        self.preprocess = preprocess
        self.wnid_to_index = wnid_to_index

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        example = self.dataset[idx]
        image = preprocess((example["image"]))
        label = wnid_to_r_index[example["wnid"]]
        return image, label

In [ ]:
r_dataset = ImageNetRDataset(ds['test'], preprocess, wnid_to_r_index)
r_dataloader = DataLoader(r_dataset, batch_size=32, num_workers=2)

In [ ]:
images, labels = next(iter(r_dataloader))
print(images.shape, labels.shape)
print(labels[:5])

### Prepare the Image Features

In [ ]:
r_image_cach = build_and_cache_image_features(model, device, r_dataloader, "./features", "imagenet-r")

### Evaluation

In [ ]:
r_image_features = r_image_cach['image_features'].to(device)
r_labels = r_image_cach['labels'].to(device)
r_text_features = r_text_features.to(device)

similarity = r_image_features @ r_text_features
logits = model.logit_scale.exp() * similarity
acc1, acc5 = top_k_accuracy(logits, r_labels, topk=(1, 5))

In [ ]:
probs = torch.softmax(logits, dim=-1)
conf, pred = probs.max(dim=-1)

bins = torch.linspace(0, 1, 11)
for i in range(10):
    mask = (conf >= bins[i]) & (conf < bins[i+1])
    if mask.sum() == 0:
        continue
    acc = (pred[mask] == r_labels[mask]).float().mean().item()
    print(f"conf [{bins[i]:.1f},{bins[i+1]:.1f}): n={mask.sum().item():4d}  acc={acc:.3f}")

## Binning Probe on TPT

In [ ]:
import torchvision.transforms as transforms

augment_transform = transforms.Compose([
    transforms.Lambda(lambda im: im.convert("RGB")),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711)) # from clip
])

In [ ]:
def generate_N_views(N, transform_fn, image):
  views = [transform_fn(image) for _ in range(N)]
  curr_img = preprocess(image)
  views.append(curr_img)
  views = torch.stack(views)

  return views

### Training Loop

In [ ]:
from coop import PromptLearner, TextEncoderWrapper

In [ ]:
prompt_learner = PromptLearner(model, device, 4, tokenizer, 512, r_class_names)
text_encoder = TextEncoderWrapper(model)

for param in model.parameters():
  param.requires_grad_(False)

print([n for n, p in prompt_learner.named_parameters() if p.requires_grad])

In [ ]:
from tqdm.notebook import tqdm
import torch.nn.functional as F
from tpt import tpt_entropy_loss

correct = 0
total = 0
subset_size = 500
logit_scale = model.logit_scale.exp()

confidences = []
correctness = []

for i in tqdm(range(subset_size)):
    test_image = ds['test'][i]['image']
    true_label = wnid_to_r_index[ds['test'][i]['wnid']]

    prompt_learner.reset_context()
    optimizer = torch.optim.AdamW(prompt_learner.parameters(), lr=0.005)

    test_image_views = generate_N_views(63, augment_transform, test_image)
    image_features = model.encode_image(test_image_views.to(device))
    prompts, tok_prompts = prompt_learner()
    text_features = text_encoder(prompts, tok_prompts)
    text_features = text_features / text_features.norm(dim=-1,keepdim=True)

    logits = image_features @ text_features.t()
    loss = tpt_entropy_loss(logits)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    with torch.no_grad():
      clean_image = preprocess(test_image).unsqueeze(0).to(device)
      clean_feat = model.encode_image(clean_image)
      clean_feat = clean_feat / clean_feat.norm(dim=-1, keepdim=True)
      prompts, tok = prompt_learner()
      text_feat = text_encoder(prompts, tok)
      text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)

      scaled_logits = logit_scale * (clean_feat @ text_feat.t())
      probs = torch.softmax(scaled_logits, dim=-1)
      conf, pred = probs.max(dim=-1)

    confidences.append(conf.item())
    correctness.append(pred.item() == true_label)
    correct += (pred.item() == true_label)
    total += 1

conf_t = torch.tensor(confidences)
correct_t = torch.tensor(correctness)

In [ ]:
accuracy = 100 * correct / total
accuracy

### Binning Probe

subset = 200;  
conf [0.00,0.50): n=  78  acc=0.333  
conf [0.50,0.70): n=  24  acc=0.708  
conf [0.70,0.85): n=  30  acc=0.800  
conf [0.85,1.00): n=  68  acc=0.971  

subset = 500;  
conf [0.00,0.50): n= 232  acc=0.289  
conf [0.50,0.70): n=  73  acc=0.740  
conf [0.70,0.85): n=  63  acc=0.810  
conf [0.85,1.00): n= 132  acc=0.947  

In [ ]:
bins = torch.tensor([0.0, 0.5, 0.7, 0.85, 1.0])

for i in range(len(bins) - 1):
    mask = (conf_t >= bins[i]) & (conf_t < bins[i+1])
    n = mask.sum().item()
    if n == 0:
        continue
    acc = correct_t[mask].float().mean().item()
    print(f"conf [{bins[i]:.2f},{bins[i+1]:.2f}): n={n:4d}  acc={acc:.3f}")

# Probe 2 - domain classifier

v2 vs sketch               real=0.991  null=0.496  gap=+0.495  
v2 vs r                    real=0.957  null=0.500  gap=+0.457  
r vs sketch                real=0.894  null=0.498  gap=+0.395  
photo vs sketch (PACS)     real=1.000  null=0.492  gap=+0.508  


In [ ]:
from clip_zeroshot import load_cached_image_features

In [ ]:
r_feat = load_cached_image_features('./features/imagenet-r.pt')
sk_feat = load_cached_image_features('./features/sk_eval_features.pt')
imgnt_v2_feats = load_cached_image_features('/content/features/imagenetv2_img_feats.pt')
pacs_photo_feats = load_cached_image_features('/content/features/pacs_photo.pt')
pacs_sketch_feats = load_cached_image_features('/content/features/pacs_sketch.pt')

In [ ]:
feat_r = r_feat['image_features']
feat_sketch = sk_feat['image_features']
feat_v2 = imgnt_v2_feats['image_features']
feat_photo = pacs_photo_feats['image_features']
feat_sketch_pacs = pacs_sketch_feats['image_features']

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

def separability(fa, fb, seed=42):
    n = min(len(fa), len(fb))
    g = torch.Generator().manual_seed(seed)
    fa = fa[torch.randperm(len(fa), generator=g)[:n]]
    fb = fb[torch.randperm(len(fb), generator=g)[:n]]
    X = torch.cat([fa, fb]).cpu().numpy()
    y = torch.cat([torch.zeros(n), torch.ones(n)]).numpy()

    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    real = LogisticRegression(max_iter=1000).fit(Xtr, ytr).score(Xte, yte)

    ysh = np.random.permutation(y)
    Xtr2, Xte2, ytr2, yte2 = train_test_split(X, ysh, test_size=0.3, random_state=seed, stratify=ysh)
    null = LogisticRegression(max_iter=1000).fit(Xtr2, ytr2).score(Xte2, yte2)
    return real, null

for name, (fa, fb) in {
    "v2 vs sketch":  (feat_v2, feat_sketch),
    "v2 vs r":       (feat_v2, feat_r),
    "r vs sketch":   (feat_r, feat_sketch),
    "photo vs sketch (PACS)": (feat_photo, feat_sketch_pacs),
}.items():
    real, null = separability(fa, fb)
    print(f"{name:26s} real={real:.3f}  null={null:.3f}  gap={real-null:+.3f}")

# TPT Selection Ablation

------Current selection: 0.05-------  
accuracy: 58.7   ECE: 7.22%  
conf [0.00,0.50): n= 141  acc=0.298  
conf [0.50,0.70): n=  41  acc=0.756  
conf [0.70,0.85): n=  39  acc=0.795  
conf [0.85,1.00): n=  79  acc=0.911  

------Current selection: 0.1-------  
accuracy: 59.7   ECE: 8.91%  
conf [0.00,0.50): n= 135  acc=0.326  
conf [0.50,0.70): n=  48  acc=0.688  
conf [0.70,0.85): n=  34  acc=0.735  
conf [0.85,1.00): n=  83  acc=0.928  

------Current selection: 0.2-------  
accuracy: 60.7   ECE: 8.20%  
conf [0.00,0.50): n= 141  acc=0.348  
conf [0.50,0.70): n=  40  acc=0.750  
conf [0.70,0.85): n=  37  acc=0.757  
conf [0.85,1.00): n=  82  acc=0.915  

In [ ]:
def compute_ece(conf_t, correct_t, n_bins=10):
    bins = torch.linspace(0, 1, n_bins + 1)
    ece = 0.0
    N = len(conf_t)
    for b in range(n_bins):
        mask = (conf_t >= bins[b]) & (conf_t < bins[b+1])
        n = mask.sum().item()
        if n == 0:
            continue
        acc = correct_t[mask].float().mean().item()
        avg_conf = conf_t[mask].mean().item()
        ece += (n / N) * abs(acc - avg_conf)
    return ece * 100

In [ ]:
from tqdm.notebook import tqdm
import torch.nn.functional as F
from tpt import tpt_entropy_loss


for frac in [0.05, 0.10, 0.20]:
  correct = 0
  total = 0
  subset_size = 300
  logit_scale = model.logit_scale.exp()

  confidences = []
  correctness = []

  for i in tqdm(range(subset_size)):
    test_image = ds['test'][i]['image']
    true_label = wnid_to_r_index[ds['test'][i]['wnid']]

    prompt_learner.reset_context()
    optimizer = torch.optim.AdamW(prompt_learner.parameters(), lr=0.005)

    test_image_views = generate_N_views(63, augment_transform, test_image)
    image_features = model.encode_image(test_image_views.to(device))
    prompts, tok_prompts = prompt_learner()
    text_features = text_encoder(prompts, tok_prompts)
    text_features = text_features / text_features.norm(dim=-1,keepdim=True)

    logits = image_features @ text_features.t()
    loss = tpt_entropy_loss(logits, top_k_fraction=frac)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    with torch.no_grad():
      clean_image = preprocess(test_image).unsqueeze(0).to(device)
      clean_feat = model.encode_image(clean_image)
      clean_feat = clean_feat / clean_feat.norm(dim=-1, keepdim=True)
      prompts, tok = prompt_learner()
      text_feat = text_encoder(prompts, tok)
      text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)

      scaled_logits = logit_scale * (clean_feat @ text_feat.t())
      probs = torch.softmax(scaled_logits, dim=-1)
      conf, pred = probs.max(dim=-1)

    confidences.append(conf.item())
    correctness.append(pred.item() == true_label)
    correct += (pred.item() == true_label)
    total += 1

  conf_t = torch.tensor(confidences)
  correct_t = torch.tensor(correctness)
  accuracy = 100 * correct / total
  ece = compute_ece(conf_t, correct_t, n_bins=10)

  print(f"------Current selection: {frac}-------")
  print(f"accuracy: {accuracy:.1f}   ECE: {ece:.2f}%")

  bins = torch.tensor([0.0, 0.5, 0.7, 0.85, 1.0])

  for b in range(len(bins) - 1):
    mask = (conf_t >= bins[b]) & (conf_t < bins[b+1])
    n = mask.sum().item()

    if n == 0:
        continue

    acc = correct_t[mask].float().mean().item()
    print(f"conf [{bins[b]:.2f},{bins[b+1]:.2f}): n={n:4d}  acc={acc:.3f}")
